# Synthetic data contracts

Investigate deterministic generation, split isolation and class imbalance.

Run with the repository virtual environment. These are synthetic research experiments; units and assumptions are explicit below.

## Inputs

No external images or weights are needed.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from autonomous_vehicles.data import CLASS_NAMES, RoadScenes
from autonomous_vehicles.perception.components import connected_components

seed = 73
size = 64
splits = {name: RoadScenes(24, size, seed, name) for name in ("train", "validation", "test")}

## Compare independent splits

Each split has its own RNG namespace. Same-index samples are not shared.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 5))
for column, (name, dataset) in enumerate(splits.items()):
    image, mask = dataset[0]
    axes[0, column].imshow(image.permute(1, 2, 0))
    axes[1, column].imshow(mask, vmin=0, vmax=2)
    axes[0, column].set_title(name)
assert not torch.equal(splits["train"][0][0], splits["test"][0][0])
plt.tight_layout()
plt.show()

## Order independence

Sample identity depends on split, seed and index, not the order of iteration.

In [ ]:
indices = [7, 2, 15, 0]
first = {index: splits["train"][index][0].clone() for index in indices}
second = {index: splits["train"][index][0] for index in reversed(indices)}
checks = [torch.equal(first[index], second[index]) for index in indices]
assert all(checks)
print({"indices": indices, "equal": checks})

## Class frequency

Count pixels across complete splits, rather than averaging per-batch fractions.

In [ ]:
fractions = {}
for name, dataset in splits.items():
    counts = torch.zeros(3)
    for _, mask in dataset:
        counts += torch.bincount(mask.flatten(), minlength=3)
    fractions[name] = (counts / counts.sum()).numpy()
fig, ax = plt.subplots(figsize=(7, 3))
for index, (name, values) in enumerate(fractions.items()):
    ax.bar(np.arange(3) + index * 0.25, values, width=0.25, label=name)
ax.set_xticks(np.arange(3) + 0.25, CLASS_NAMES)
ax.set_ylabel("Pixel fraction")
ax.legend()
plt.show()

## Instance ambiguity

Connected components merge touching objects. Pixel semantics alone cannot guarantee instance identities.

In [ ]:
component_counts = []
areas = []
for _, mask in splits["train"]:
    components = connected_components(mask.numpy(), minimum_pixels=1)
    component_counts.append(len(components))
    areas.extend(component.pixels for component in components)
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].hist(component_counts, bins=np.arange(6) - 0.5)
axes[0].set_xlabel("Obstacle components per scene")
axes[1].hist(areas, bins=12)
axes[1].set_xlabel("Component area (pixels)")
plt.show()
print("Inspect other seeds and sizes before claiming distribution coverage.")

## Research record

Record the seed, configuration, dependency lock and observed failures with each experiment. Results here demonstrate the implemented model under its stated assumptions. They do not establish performance outside this synthetic distribution.